# HigherCriticismSelector

Scores each feature with the two-sided Mann-Whitney U test, sorts the
p-values, and keeps every feature up to the rank where the Higher Criticism
score peaks. It picks its own count.

- **Cost:** one rank test per feature, about 10 s on 1,500 samples x 28,500
  features.
- **Strength:** built for many weak signals hidden among many null features.
- **Weakness:** it searches only the smallest `alpha0` share of p-values, and
  assumes independent features.
- **Modes:** `variant='2008'` (default) or `'2004'`. Two classes only.

The U test uses only the order of each feature's values, as an AUC does, so
any monotone transform of a feature leaves the selection unchanged.

In [1]:
import numpy as np

from featsel.selectors import HigherCriticismSelector


def rare_weak(n_signal, shift, n_samples=300, n_features=2000, seed=0):
    """Two classes; the first n_signal features differ in mean by shift."""
    rng = np.random.default_rng(seed)
    y = np.repeat([0, 1], n_samples // 2)
    X = rng.normal(size=(n_samples, n_features))
    X[y == 1, :n_signal] += shift
    return X, y


def report(X, y, n_signal, **params):
    chosen = set(HigherCriticismSelector(**params).fit(X, y).get_support(indices=True))
    truth = set(range(n_signal))
    print(f'{params}: kept {len(chosen):4d}: '
          f'{len(chosen & truth)} of {n_signal} signal, {len(chosen - truth)} false')

## Rare, weak signal: what HC is built for

40 of 2,000 features carry a small shift. HC keeps most of them, and accepts
some false features as the price: it is designed to miss few.

In [2]:
X, y = rare_weak(n_signal=40, shift=0.4)
for variant in ('2008', '2004'):
    report(X, y, 40, variant=variant)

{'variant': '2008'}: kept   79: 34 of 40 signal, 45 false
{'variant': '2004'}: kept    1: 1 of 40 signal, 0 false


The 2004 variant divides by sqrt(p(i)), so the very smallest p-values
dominate and it stops early. The 2008 variant scales by the expected
position i/(N+1) instead, and goes further.

## Only the order matters

A monotone transform of every feature changes nothing.

In [3]:
raw = HigherCriticismSelector().fit(X, y).get_support()
transformed = HigherCriticismSelector().fit(np.exp(X) ** 3, y).get_support()
print('same selection:', np.array_equal(raw, transformed))

same selection: True


## Where it stops: the alpha0 edge

When signal is not rare, the HC score is still rising at the edge of the
search range, the smallest `alpha0` = 10% of p-values. HC then keeps exactly
that many features. Here 600 of 2,000 features carry a strong shift, and HC
keeps 200. SCAN-B behaves like this: thousands of correlated genes differ
between subtypes, and HC keeps exactly 10% of them (see
`docs/report/03_data.md`).

In [4]:
X_dense, y_dense = rare_weak(n_signal=600, shift=1.0)
report(X_dense, y_dense, 600, alpha0=0.1)

{'alpha0': 0.1}: kept  200: 200 of 600 signal, 0 false
